# Dataset, alignment and profitability targets

The research ranks events at each scheduled scan to allocate a limited monitoring budget. The primary target is simulated net P&L over the following window. Opportunity occurrence remains a secondary target. This notebook inspects raw books, causal alignment, snapshot predictors and complete/unknown labels. It reads development data only.

In [1]:
from pathlib import Path
import os
import sys
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/config.py').exists())
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 20)
from src.reporting.workflow import profitability_dataset, profitability_run, execution_path
from src.paths import PROCESSED_DIR
from src.config import SPEC, ScreeningSpec, for_window
from src.data.universe import build_plan
from src.data.dataset import align_books
from src.data.metadata import add_metadata
from src.data.snapshots import prepare
from src.data.profitability import build_profitability, load_profitability
from src.reporting.workflow import (dataset_paths, dataset_spec, timestamp,
    new_profitability_dataset, select_profitability_dataset)


## Rebuild controls

The default reads completed local results. `REBUILD = True` constructs fresh targets for every event-hour, independently of model selections. `WINDOW_MINUTES` sets both scan cadence and monitoring duration. A changed window first rebuilds its scheduled snapshot inputs. The profitability builder updates quantity-dependent quote features and their lags to full uncapped depth.

Execution uses 100 ms assumed arrival, no artificial cash/share cap, and a one-second partial-basket unwind/hold. Losses remain in targets; incomplete coverage and unpriceable attempted outcomes remain unknown. A completed rebuild becomes the dataset used by notebook 02.

In [2]:
REBUILD = False
WINDOW_MINUTES = 60
if REBUILD:
    spec = for_window(WINDOW_MINUTES)
    features_path, _ = dataset_paths()
    if dataset_spec(features_path).cadence_minutes != WINDOW_MINUTES:
        feature_dir = PROCESSED_DIR / 'profitability_inputs' / timestamp()
        prepare(feature_dir, top_n=5, spec=spec)
        features_path = feature_dir / 'hourly_features.parquet'
    destination = new_profitability_dataset()
    build_profitability(destination, features_path, spec)
    select_profitability_dataset(destination)
dataset_dir = profitability_dataset()
design = json.loads((dataset_dir / 'design.json').read_text())
spec = ScreeningSpec(**design['contract'])
print('Dataset:', dataset_dir)
print('Monitoring interval:', spec.horizon_minutes, 'minutes')
display(pd.Series(design['execution'], name='Execution configuration'))


Dataset: C:\Users\Daron\coding\inefficiency-prediction\data\processed\profitability
Monitoring interval: 60 minutes


wait_s                         0.0
delay_s                        0.1
sizing                    adaptive
min_entry_gross_usd            0.0
freshness_s                   None
max_quantity                  None
cash_cap_usd                  None
unwind_delay_s                 1.0
min_quantity                     5
min_limit_margin_usd           0.0
monitoring_minutes              60
compute_failure_stress       False
Name: Execution configuration, dtype: object

## Raw token order books

Each token has its own asynchronous snapshots. Asks contain available purchase prices and sizes; bids contain prices and sizes available when selling owned tokens. Source timestamps determine the primary replay clock. Provider indexing timestamps resolve duplicate source-time records; they do not establish measured delivery latency.

In [3]:
plans, _ = build_plan('all')
plan = plans[0]
frames = {token: pd.read_parquet(path, filters=[
    ('snapshot_time', '>=', pd.Timestamp(SPEC.development_start)),
    ('snapshot_time', '<', pd.Timestamp(SPEC.lockbox_start))])
    for token, path in list(plan['paths'].items())[:2]}
raw = next(iter(frames.values()))
display(raw.head(3))
display(raw.dtypes.rename('dtype'))
display(pd.DataFrame(json.loads(raw.asks_json.iloc[0])).head(8))


,token_id,snapshot_time,snapshot_timestamp_ms,indexed_at_time,indexed_at_ms,market_hash,asset_id,tick_size,min_order_size,orderbook_neg_risk,bids_json,asks_json
0,1082944184701543310123155764189258968225550889...,2025-10-14 19:55:22.539000+00:00,1760471722539,2025-10-14 19:55:25.139000+00:00,1760471725139,0x7a461888f0011b1730d74422388d76626b800033eb10...,1082944184701543310123155764189258968225550889...,0.01,5.0,True,"[{""size"": ""233"", ""price"": ""0.01""}, {""size"": ""5...","[{""size"": ""5444"", ""price"": ""0.99""}, {""size"": ""..."
1,1082944184701543310123155764189258968225550889...,2025-10-14 19:55:28.025000+00:00,1760471728025,2025-10-14 19:55:30.023000+00:00,1760471730023,0x7a461888f0011b1730d74422388d76626b800033eb10...,1082944184701543310123155764189258968225550889...,0.01,5.0,True,"[{""size"": ""233"", ""price"": ""0.01""}, {""size"": ""5...","[{""size"": ""5444"", ""price"": ""0.99""}, {""size"": ""..."
2,1082944184701543310123155764189258968225550889...,2025-10-14 20:03:09.378000+00:00,1760472189378,2025-10-14 20:03:13.463000+00:00,1760472193463,0x7a461888f0011b1730d74422388d76626b800033eb10...,1082944184701543310123155764189258968225550889...,0.01,5.0,True,"[{""size"": ""233"", ""price"": ""0.01""}, {""size"": ""5...","[{""size"": ""5444"", ""price"": ""0.99""}, {""size"": ""..."


token_id                                 str
snapshot_time            datetime64[ms, UTC]
snapshot_timestamp_ms                  int64
indexed_at_time          datetime64[ms, UTC]
indexed_at_ms                          int64
market_hash                              str
asset_id                                 str
tick_size                            float64
min_order_size                       float64
orderbook_neg_risk                      bool
bids_json                                str
asks_json                                str
Name: dtype, dtype: object

,size,price
0,5444,0.99
1,2800,0.98
2,2000,0.97
3,231,0.89
4,1500,0.8
5,1332,0.79
6,388.35,0.77
7,100,0.76


## Backward-as-of alignment

At each timestamp, use the latest preceding snapshot of each token. Never interpolate or use a future row. For labels and fills, each token is bounded by its own recording extent. Scheduled features use only the current and preceding scheduled snapshots.

In [4]:
times = pd.DatetimeIndex([], tz='UTC')
for frame in frames.values():
    times = times.union(pd.DatetimeIndex(frame.snapshot_time))
aligned = align_books(frames, times.sort_values()[:6])
display(aligned)
assert (aligned.loc[aligned.observed, 'source_time'] <= aligned.loc[aligned.observed, 'time']).all()


,time,token_id,source_time,age_s,observed,clean,best_bid,best_ask,ask_depth
0,2025-10-14 19:55:22.539000+00:00,1082944184701543310123155764189258968225550889...,2025-10-14 19:55:22.539000+00:00,0.000,True,True,0.170,0.20,21455.92
1,2025-10-14 19:55:22.539000+00:00,4793269844871547190601025477822734255907377069...,NaT,NaN,False,False,NaN,NaN,NaN
2,2025-10-14 19:55:28.025000+00:00,1082944184701543310123155764189258968225550889...,2025-10-14 19:55:28.025000+00:00,0.000,True,True,0.170,0.20,20113.92
3,2025-10-14 19:55:28.025000+00:00,4793269844871547190601025477822734255907377069...,NaT,NaN,False,False,NaN,NaN,NaN
4,2025-10-14 20:03:09.378000+00:00,1082944184701543310123155764189258968225550889...,2025-10-14 20:03:09.378000+00:00,0.000,True,True,0.170,0.20,27011.92
5,2025-10-14 20:03:09.378000+00:00,4793269844871547190601025477822734255907377069...,NaT,NaN,False,False,NaN,NaN,NaN
6,2025-10-14 20:07:55.941000+00:00,1082944184701543310123155764189258968225550889...,2025-10-14 20:03:09.378000+00:00,286.563,True,True,0.170,0.20,27011.92
7,2025-10-14 20:07:55.941000+00:00,4793269844871547190601025477822734255907377069...,2025-10-14 20:07:55.941000+00:00,0.000,True,True,0.013,0.09,15654.58
8,2025-10-14 20:08:15.884000+00:00,1082944184701543310123155764189258968225550889...,2025-10-14 20:08:15.884000+00:00,0.000,True,True,0.180,0.21,26250.46
9,2025-10-14 20:08:15.884000+00:00,4793269844871547190601025477822734255907377069...,2025-10-14 20:07:55.941000+00:00,19.943,True,True,0.013,0.09,15654.58


## One modelling row per event and scheduled scan

There are 11 current book features, 11 preceding-scan values and 11 changes. Quantity-dependent features use uncapped integer depth. Metadata adds event type, broad topic and age since the earliest frozen member market was created. Models compare book-only and combined inputs; outcome fields never enter predictors.

`pnl_target` is the sum of simulated net returns from attempts initiated in the window, including exits after its boundary. `activity_target` indicates any attempted trade. `target` indicates at least one positive attempt, which does not necessarily imply a profitable hour. All three are missing when the window is not fully evaluable.

In [5]:
examples = add_metadata(load_profitability(dataset_dir))
feature_columns = [c for c in examples if c.startswith('x_')]
display(pd.DataFrame({'book_feature': feature_columns}))
display(examples[['event_slug','time','x_best_quoted_net_usd','x_ask_depth_sum','x_book_age_max_s',
    'm_event_type','m_topic','m_age_days','coverage_fraction','label_known','target','activity_target','pnl_target']].head(8))
display(examples.groupby('stratum').agg(events=('event_slug','nunique'),candidate_windows=('time','size'),
    known=('label_known','sum'),trading_hours=('activity_target','sum')))
known = examples.loc[examples.label_known]
display(pd.Series({'positive_net_hours': int(known.pnl_target.gt(0).sum()),
    'zero_net_hours': int(known.pnl_target.eq(0).sum()),'negative_net_hours': int(known.pnl_target.lt(0).sum()),
    'unknown_hours': int((~examples.label_known).sum()),
    'positive_attempt_but_nonpositive_hour': int((known.target.eq(1) & known.pnl_target.le(0)).sum())}))
display(known.pnl_target.describe(percentiles=[.5,.9,.99,.999]))
assert examples.loc[~examples.label_known, ['pnl_target','activity_target','target']].isna().all().all()
assert known.loc[known.activity_target.eq(0),'pnl_target'].eq(0).all()


,book_feature
0,x_active_packages
1,x_active_tokens
2,x_seen_fraction
3,x_clean_fraction
4,x_spread_mean
5,x_spread_max
6,x_ask_depth_sum
7,x_book_age_max_s
8,x_best_quoted_net_usd
9,x_size_feasible_package_fraction


,event_slug,time,x_best_quoted_net_usd,x_ask_depth_sum,x_book_age_max_s,m_event_type,m_topic,m_age_days,coverage_fraction,label_known,target,activity_target,pnl_target
0,fed-decision-in-october,2025-10-14 20:00:00+00:00,-0.09500,30083761.97,269.399,negrisk,central_bank,117.145896,1.0,True,0.0,0.0,0.0
1,largest-company-end-of-november,2025-10-14 20:00:00+00:00,-0.15000,41606882.99,721.115,negrisk,technology_business,3.954294,1.0,True,0.0,0.0,0.0
2,world-series-champion-2025,2025-10-14 20:00:00+00:00,-0.11000,37802242.61,844.702,negrisk,sports,230.057416,1.0,True,0.0,0.0,0.0
3,elon-musk-of-tweets-october-17-october-24,2025-10-14 21:00:00+00:00,-2.61500,261883.09,205.799,negrisk,tweet_counts,0.708290,1.0,True,0.0,0.0,0.0
4,fed-decision-in-october,2025-10-14 21:00:00+00:00,-0.09500,30094356.46,257.047,negrisk,central_bank,117.187562,1.0,True,0.0,0.0,0.0
5,largest-company-end-of-november,2025-10-14 21:00:00+00:00,-0.30000,41615208.69,1392.349,negrisk,technology_business,3.995960,1.0,True,0.0,0.0,0.0
6,where-will-trump-and-putin-meet-next-393,2025-10-14 21:00:00+00:00,-0.25500,432281.11,522.188,negrisk,geopolitics,56.868839,1.0,True,0.0,0.0,0.0
7,world-series-champion-2025,2025-10-14 21:00:00+00:00,-0.06418,37808456.28,1687.358,negrisk,sports,230.099082,1.0,True,0.0,0.0,0.0


,events,candidate_windows,known,trading_hours
stratum,,,,
ladder,5,3320,1849,42.0
negrisk,43,23515,19978,412.0


positive_net_hours                         437
zero_net_hours                           21373
negative_net_hours                          17
unknown_hours                             5008
positive_attempt_but_nonpositive_hour        7
dtype: int64

count    21827.000000
mean         0.275663
std          7.491204
min       -124.377200
50%          0.000000
90%          0.000000
99%          0.500000
99.9%       77.964842
max        710.340520
Name: pnl_target, dtype: float64

Standalone training windows begin with an empty reservation ledger. Their P&L is a monitoring-value target, not an additive strategy return. Notebook 03 preserves reservations across selected windows and folds. Unknown event-hours remain in the ranking pool and consume monitoring slots. Continue with [model comparison](02_models.ipynb).

## Discover a new market universe

This optional metadata-only step classifies a supplied export and generates supported basket definitions. It writes a separate manifest; the frozen dataset and model run above are unchanged. Input fields and supported contract grammars are documented in `docs/dataset.md`. Point-in-time mode requires metadata availability timestamps. Use frozen mode only when retrospective metadata assumptions are explicitly intended. Rebuilding models for a changed universe requires a separate dataset build.

In [ ]:
from src.data.universe_detection import (
    detect_universe, normalise_market, normalise_event, read_export,
)

RUN_DISCOVERY = False
MARKET_EXPORT = ROOT / "data/metadata/new_markets.jsonl"
EVENT_EXPORT = ROOT / "data/metadata/new_events.jsonl"
METADATA_MODE = "point_in_time"

if RUN_DISCOVERY:
    discovered = detect_universe(
        read_export(MARKET_EXPORT, normalise_market),
        read_export(EVENT_EXPORT, normalise_event),
        window_start=SPEC.development_start,
        window_end=SPEC.lockbox_start,
        metadata_mode=METADATA_MODE,
    )
    destination = PROCESSED_DIR / "universe_detection" / timestamp()
    discovered.write(destination)
    print("Discovery manifest:", destination)
    display(discovered.events)
    display(discovered.markets.groupby(["status", "reason"]).size().rename("markets"))
    display(discovered.pairs.head(20))
